# Decompose server (Grounding DINO + SAM2 + LaMa) behind a Cloudflare quick tunnel
Settings: **Accelerator = GPU T4**, **Internet = On**. Run the cells in order; the last one keeps the session alive. Stop the notebook to shut the backend down.

In [ ]:
import os, subprocess, sys
os.environ['SAM2_BUILD_CUDA'] = '0'
for pkg in (['git+https://github.com/facebookresearch/sam2.git'], ['--no-deps', 'simple-lama-inpainting'], ['fire'], ['fastapi', 'uvicorn'], ['diffusers==0.37.0']):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg, capture_output=True, text=True)
    print(pkg, 'rc', r.returncode)

In [ ]:
import base64, io, zipfile
os.makedirs('/kaggle/working/code', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall('/kaggle/working/code')
sys.path.insert(0, '/kaggle/working/code/prototype')
from layerproto.backends.sam2_decomposer import Sam2Decomposer
dec = Sam2Decomposer(device='cuda')
print('models loaded')

In [ ]:
import json, re, secrets, threading, time, urllib.request, uvicorn
from layerproto.remote_server import build_app

TOKEN = secrets.token_urlsafe(16)
stub = type('NoChain', (), {'info': lambda self: {'backend': 'decompose-only'}})()   # chain routes are not served here
server = uvicorn.Server(uvicorn.Config(build_app(stub, TOKEN, decomposer=dec), host='127.0.0.1', port=8765, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
if not os.path.exists('/kaggle/working/cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '/kaggle/working/cloudflared')
    os.chmod('/kaggle/working/cloudflared', 0o755)
tunnel = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8765'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com', line)
    if m:
        url = m.group(0); break
print('Put this into prototype/config.json on the laptop (keep image_backend as it is):')
print(json.dumps({'remote_url': url, 'remote_token': TOKEN}, indent=1))

In [ ]:
# keep the session busy; stop this cell or the notebook to shut the backend down
while True:
    time.sleep(60)